In [92]:
import ee
ee.Initialize(project='sig-ee-cloud')
import datetime
import scripts.analysis_functions as af
import scripts.utils as utils
#individual fire export uses geetools, currently not working
#import geetools
%load_ext autoreload
%autoreload 2

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


### Run Burn Severity for a particular year

Dev Note: 

even though there seems to have been attempts of handling edge cases where a composite date window returns no images (see `af.getLandsatToaRobust()`), I still got an error " 'NIR' band not available" due to one specific fire's pre fire img collection containing no Landsat imagery. 

I figured out which fire it was in the DEBUG section below (see `af.debug_img_collections_*` functions). Even though no real Landsat images were returned in the pre image collection date filter, the result of `getLandsatToaRobust()` with that pre fire window returns one image for the problem fire. However, inspecting the bands list of the problem fire's computed image composite returns an empty list, compared to all other fires' pre/post fire composites which return 25 bands. Therefore the error can be traced back to `gic2.doIndices()` not being able to compute any indices requiring the "NIR" band. Just noting this as a known issue that we might want to solve for good if we have time/motivation.

#### Production run with NIFC

In [62]:
# For running a year of fires from NIFC for burn severity
#  Based the corrections in the simulation code, but no fire filter dates, 
#   and replaced 'simulation' date with run date (the current date)
# REQUIRES USER INPUT/SELECTION, see starred section below. 

# For FFv4, run TWICE! Once for all 2023 fires, and once for 2024 fires up to current. 
# Could have written loop, instead just ran twice. 

# Last edited 2024-10-14 by dnekorchuk

#***
# USER: 
# 1. Set data import, align year and import dataset
# 2. Change data_origin filename tag and asset out folder as needed
# 3. Scroll down and set export flags
# 4. Select window type
#***

#### IMPORT DATA ******************************************

## CHANGE BETWEEN 2023 and 2024 to run both. 

## NIFC 
fc_list = [
    # "projects/pyregence-ee/assets/conus/nifc/delos-fuels-run-2025/nifc_fires_2020_gte100ac_20250613",
    # "projects/pyregence-ee/assets/conus/nifc/delos-fuels-run-2025/nifc_fires_2021_gte100ac_20250612",
    # "projects/pyregence-ee/assets/conus/nifc/delos-fuels-run-2025/nifc_fires_2022_gte100ac_20250612",
    # "projects/pyregence-ee/assets/conus/nifc/delos-fuels-run-2025/nifc_fires_2023_gte100ac_20250612",
    "projects/pyregence-ee/assets/conus/nifc/delos-fuels-run-2025/nifc_fires_2024_gte100ac_20250612"
]

for asset in fc_list:
    print(f'Importing FeatureCollection: {asset}')
    try:
        fires_raw = ee.FeatureCollection(asset)
        ee.data.getAsset(asset)
    except:
        raise FileNotFoundError(f'Error importing FeatureCollection: {asset}. Check if it exists or is accessible.')
    yr = asset.split('_')[-3]
    print(f'Processing fires for year: {yr}')



    tag = 'gte100ac'
    #relevant field names: Acres, Discovery, GlobalID, Name
    name_field = 'Name'
    id_field = 'GlobalID'
    size_field = 'Acres'

    #For testing subsets of fires
    # id_list = ['dd114282-a54d-414b-bdb1-4e180fc1ea7d', '221b33e5-594c-4f68-9131-9d412e057dfd']
    # fires_known = fires_raw.filter(ee.Filter.inList(id_field,id_list))
    # #plus some other fires
    # fires_add = fires_raw.limit(22, 'Discovery', False) 
    # fires = fires_known.merge(fires_add)

    #For run with all fires
    fires = fires_raw

    data_origin = 'nifc'
    asset_folder = 'projects/pyregence-ee/assets/conus/nifc/delos-fuels-run-2025' #'projects/pyregence-ee/assets/fires_bs_tool/dmn_testing'


    ## USER SETTINGS **********************************

    export_flag = False #severity classes, composite/mosiac conus
    per_fire_export = False #indiv fire results for dNBR and severity. #trying fitoprincipe batch export, failing
    csv_export_flag = False #csv log/diagnostic dates and info
    #Variants of recent fire windows
    # 'fixed' - fixed 90 days post fire, but won't go past sim date
    # 'expanding' - variable days expanding from fire to current/sim date
    # 'sliding' - fixed 90 day window, sliding along to most recent 90 days to current/sim date.  
    recent_type = 'sliding' #FFv4 using same sliding setting as FFv3

    ##### SET RUN DATE #####

    #Today
    run_date = datetime.datetime.now()
    run_date_ee = ee.Date(run_date)
    run_date_formatted = str(run_date)[0:10]

    # Map over the fires, adding the run date 
    #  note: must use subfunction set_windows_sim() in bs calc function
    #  Also add id field for labeling bands in exported intermediate products
    def set_run_date(feat: ee.Feature):
        fire_date = ee.Date(feat.getString('Discovery'))
        #calculate number days from Discovery to run date (negative is before run date)
        days_from_run = fire_date.difference(run_date_ee,'day')
        #Create a name for the individual fire export. Not currently used until indiv export fixed.
        #concatenate name pieces. Note: cannot start with number, if doing band-method. 
        fire_id_prefix = ee.String('Fire_')
        fire_name_raw = ee.String(feat.get(name_field))
        # cannot have () and probably other special characters in band names, also removes all whitespace
        fire_name = fire_name_raw.replace('[^a-zA-Z0-9]', "", 'g')
        sep = ee.String('_')
        fire_event_id = ee.String(feat.get(id_field))
        fire_id = fire_id_prefix.cat(fire_name).cat(sep).cat(fire_event_id)
        return feat.set('run_date',run_date_formatted, 
                        'days_from_run',days_from_run,
                        'fire_id',fire_id,
                        'recent_type',recent_type)
    fires = fires.map(set_run_date)

    #Fires with discovery date one day before today's date will fail unpleasantly with errors about missing date ranges
    fires = fires.filter(ee.Filter.lt('days_from_run', -2))
    
    ##### METADATA COLLECTION 1 #####
    # Metadata collection and print
    fires_count = fires.size().getInfo()
    print(f'Total Fires in FeatureCollection: {fires_count}')

    metadata = {
        'run_date':run_date.strftime('%Y%m%d'),
        'fire_count':fires_count,
        'code_origin':'BS_Mapper_FF4',
        'recent_type':recent_type
    }
    #print(metadata)


    ###### GET FIRE WINDOW DIAGNOSTICS ######

    bs_windows = ee.FeatureCollection(fires.map(af.bs_get_windows_ff3)) #can use ff3 version here, no changes

    desc_windows = f'bs_{data_origin}_{yr}_{tag}_{recent_type}_{run_date.strftime("%Y%m%d")}'
    #wanted properties. Event_ID only in MTBS, will be empty in NIFC runs
    wanted_cols = [
        id_field, name_field, size_field, #'Event_ID',
        'run_date', 'days_from_run', 'Discovery', 
        'mode', 'recent_type', 'pre_start', 'pre_end', 'post_start', 'post_end', 'double_check']
    task_csv = ee.batch.Export.table.toDrive(
        collection=bs_windows,
        description=desc_windows,
        selectors=wanted_cols,
        # folder='Fire',
        fileFormat='CSV')

    if csv_export_flag == True:
        print(f'export task started: fire window diagnostics {desc_windows} to Google drive of user.')
        # task_csv.start()


    ##### RUN BURN SEVERITY #####

    # Returns RdNBR & BS for each fire ee.Feature
    bs_coll_combined = ee.FeatureCollection(fires).map(af.bs_calc_ff3) #can use ff3 version here, no changes

    # Separate out RdNBR and Miller Threshold severity classes
    #bs_rdnbr = ee.ImageCollection(bs_coll_combined).select('RdNBR')
    bs_coll = ee.ImageCollection(bs_coll_combined).select('MillersThresholds')


    ##### Export intermediate: individual features #####

    if per_fire_export == True:

        print('Exporting individual fire results')
        desc_indiv = f'bs_{yr}_{data_origin}_recent{recent_type}_perfire_{run_date.strftime("%Y%m%d")}'
        
        # TODO: batch export isn't picking up the path correctly??
        # Parameter "name" value "projects/pyregence-ee/assets" does not match the pattern "^projects/[^/]+/assets/.*$"
        
        path_coll = asset_folder+"/"+desc_indiv #f'{asset_folder}/{desc_indiv}'
        print(f'individual fire image collection: {path_coll}') 

        #use geetools to batch export to an image collection
        for i in range(fires_count):
            img = ee.Image(ee.List(ee.ImageCollection(bs_coll_combined).toList(1, i)).get(0))
            utils.exportImgtoAsset(img, 
                        desc=f"fire_2024_bs_{i}",
                        region=None,
                        asset_folder=asset_folder, 
                        export_type='single_fire',
                        export=export_flag)
            break


    else:
        print('Skipping individual fire export')


    ###### METADATA COLLECTION 3 ######

    #fire_calc_count = ee.ImageCollection(bs_coll_combined).size() # always an image, but some are no data
    # Instead add up flag (0/1) counts in 'post_fire_calc'
    fire_calc_count = ee.ImageCollection(bs_coll_combined).aggregate_sum('post_fire_calc')
    metadata['fire_count_results'] = fire_calc_count

    ##### COMPOSITE BURN SEVERITY & FINAL EXPORT #####

    # composite that ee.ImageCollection with a max() reducer, add metadata
    bs_composite = bs_coll.max().rename('SEVERITY').set('Year',int(yr)).set(metadata)

    # To Asset
    desc = f'{data_origin}_bs_{yr}_{tag}_{recent_type}_{run_date.strftime("%Y%m%d")}'


    utils.exportImgtoAsset(bs_composite, 
                        desc=desc,
                        region=None,
                        asset_folder=asset_folder, 
                        export_type='conus',
                        export=export_flag)



Importing FeatureCollection: projects/pyregence-ee/assets/conus/nifc/delos-fuels-run-2025/nifc_fires_2024_gte100ac_20250612
Processing fires for year: 2024
Total Fires in FeatureCollection: 1349
Skipping individual fire export
would export to projects/pyregence-ee/assets/conus/nifc/delos-fuels-run-2025/nifc_bs_2024_gte100ac_sliding_20250617
set export = True to when ready


# DEBUGGING

In [93]:
from pprint import pprint
fires_debug = ee.FeatureCollection(fc_list[-1])
print(fires_debug.size().getInfo())
pprint(fires_debug.first().getInfo()['properties'])

1349
{'Acres': 296.0465335192943,
 'Discovery': '2024-03-29',
 'GlobalID': '782e2dd6-225d-4d1b-a6a9-c3c7011fe9f1',
 'Name': 'BNK 109b',
 'OBJECTID': 31815,
 'State': 'US-AL'}


In [ ]:
bands_cnt = fires_debug.map(set_run_date).map(af.debug_img_collections_comp)
pprint(bands_cnt.first().getInfo()['properties'])

{'Acres': 296.0465335192943,
 'Discovery': '2024-03-29',
 'GlobalID': '782e2dd6-225d-4d1b-a6a9-c3c7011fe9f1',
 'Name': 'BNK 109b',
 'OBJECTID': 31815,
 'State': 'US-AL',
 'days_from_run': -445.50542547453705,
 'fire_id': 'Fire_BNK109b_782e2dd6-225d-4d1b-a6a9-c3c7011fe9f1',
 'post_comp_bands': 25,
 'pre_comp_bands': 25,
 'recent_type': 'sliding',
 'run_date': '2025-06-17'}


In [ ]:
empty_comps = bands_cnt.filter(ee.Filter.Or(ee.Filter.lt('pre_comp_bands',1),
                                             ee.Filter.lt('post_comp_bands',1)))
# print(empty_comps.size().getInfo())
# pprint(empty_comps.first().getInfo()['properties'])
task = ee.batch.Export.table.toAsset(
    collection=empty_comps,
    description='bs_mapper_empty_comps',
    assetId='projects/pyregence-ee/assets/conus/nifc/delos-fuels-run-2025/bs2024_empty_comps',
    )
task.start()

In [ ]:
# set run date
fd = fires.map(set_run_date)
pprint(fd.first().getInfo()['properties'])

{'Acres': 296.0465335192943,
 'Discovery': '2024-03-29',
 'GlobalID': '782e2dd6-225d-4d1b-a6a9-c3c7011fe9f1',
 'Name': 'BNK 109b',
 'OBJECTID': 31815,
 'State': 'US-AL',
 'days_from_run': -445.50542547453705,
 'fire_id': 'Fire_BNK109b_782e2dd6-225d-4d1b-a6a9-c3c7011fe9f1',
 'recent_type': 'sliding',
 'run_date': '2025-06-17'}


In [95]:
# set the date windows 
fd = fires.map(af.bs_get_windows_ff3)
pprint(fd.first().getInfo()['properties'])

{'Acres': 296.0465335192943,
 'Discovery': '2024-03-29',
 'GlobalID': '782e2dd6-225d-4d1b-a6a9-c3c7011fe9f1',
 'Name': 'BNK 109b',
 'OBJECTID': 31815,
 'State': 'US-AL',
 'days_from_run': -445.50542547453705,
 'double_check': 'historical',
 'fire_id': 'Fire_BNK109b_782e2dd6-225d-4d1b-a6a9-c3c7011fe9f1',
 'mode': 'historical',
 'post_end': {'type': 'Date', 'value': 1743206400000},
 'post_end_readable': '20250329',
 'post_start': {'type': 'Date', 'value': 1735430400000},
 'post_start_readable': '20241229',
 'pre_end': {'type': 'Date', 'value': 1680048000000},
 'pre_end_readable': '20230329',
 'pre_start': {'type': 'Date', 'value': 1672272000000},
 'pre_start_readable': '20221229',
 'recent_type': 'sliding',
 'run_date': '2025-06-17'}


In [96]:
fd1 = fd.filter(ee.Filter.eq("Name","Twisted Arrow")).first()
pprint(fd1.getInfo()['properties'])

{'Acres': 1365.2631007770758,
 'Discovery': '2024-04-07',
 'GlobalID': 'f4a97406-7317-4a57-9a85-7773e0442891',
 'Name': 'Twisted Arrow',
 'OBJECTID': 25344,
 'State': 'US-ND',
 'days_from_run': -436.50542547453705,
 'double_check': 'historical',
 'fire_id': 'Fire_TwistedArrow_f4a97406-7317-4a57-9a85-7773e0442891',
 'mode': 'historical',
 'post_end': {'type': 'Date', 'value': 1743984000000},
 'post_end_readable': '20250407',
 'post_start': {'type': 'Date', 'value': 1736208000000},
 'post_start_readable': '20250107',
 'pre_end': {'type': 'Date', 'value': 1680825600000},
 'pre_end_readable': '20230407',
 'pre_start': {'type': 'Date', 'value': 1673049600000},
 'pre_start_readable': '20230107',
 'recent_type': 'sliding',
 'run_date': '2025-06-17'}


In [86]:
import scripts.get_image_collections2 as gic2
import scripts.analysis_functions as af
fd1_d = ee.ImageCollection(af.debug_img_collections_comp(fd1))
pprint(fd1_d.getInfo()['properties'])
# pprint(fd1_d.limit(1,"system:index",False).first().bandNames().getInfo())

{'Acres': 1365.2631007770758,
 'Discovery': '2024-04-07',
 'GlobalID': 'f4a97406-7317-4a57-9a85-7773e0442891',
 'Name': 'Twisted Arrow',
 'OBJECTID': 25344,
 'State': 'US-ND',
 'days_from_run': -436.50542547453705,
 'double_check': 'historical',
 'fire_id': 'Fire_TwistedArrow_f4a97406-7317-4a57-9a85-7773e0442891',
 'mode': 'historical',
 'post_comp_bands': 25,
 'post_end': {'type': 'Date', 'value': 1743984000000},
 'post_end_readable': '20250407',
 'post_start': {'type': 'Date', 'value': 1736208000000},
 'post_start_readable': '20250107',
 'pre_comp_bands': 0,
 'pre_end': {'type': 'Date', 'value': 1680825600000},
 'pre_end_readable': '20230407',
 'pre_start': {'type': 'Date', 'value': 1673049600000},
 'pre_start_readable': '20230107',
 'recent_type': 'sliding',
 'run_date': '2025-06-17'}
